# 🎬 AI Video Dubbing
1. Crea una Google Sheet llamada **"Doblajes"**
2. Columna A: `URL` | Columna B: `Estado` | Columna C: `Link Drive`
3. Pega un link de YouTube en **A2**
4. Dale **PLAY** a la celda de abajo (una sola vez)
5. El video aparece en **Google Drive > VideosTraducidos**

In [ ]:
#@title ▶️ DALE PLAY AQUI - Todo se hace solo
#@markdown Voz para el doblaje:
VOZ = "es-MX-JorgeNeural" #@param ["es-MX-JorgeNeural", "es-MX-DaliaNeural", "es-ES-AlvaroNeural", "es-AR-TomasNeural"]
#@markdown Nombre de tu Google Sheet:
SHEET_NAME = "Doblajes" #@param {type:"string"}

# ============================================================
# PASO 1: INSTALAR TODO
# ============================================================
print("⏳ [1/7] Instalando dependencias...")
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp', 'openai-whisper', 'edge-tts', 'gspread', 'google-auth', 'pydub'], check=True, capture_output=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], capture_output=True)
print("✅ [1/7] Dependencias listas")

# ============================================================
# PASO 2: CONECTAR GOOGLE
# ============================================================
print("⏳ [2/7] Conectando a Google...")
from google.colab import auth, drive
import gspread
from google.auth import default
import os, json, shutil, time, asyncio

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive.mount('/content/drive', force_remount=False)

DRIVE_OUTPUT = '/content/drive/MyDrive/VideosTraducidos'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print("✅ [2/7] Google conectado")

# ============================================================
# PASO 3: LEER SHEET
# ============================================================
print(f"⏳ [3/7] Leyendo Sheet '{SHEET_NAME}'...")
try:
    sheet = gc.open(SHEET_NAME).sheet1
    all_rows = sheet.get_all_values()
    pending = []
    for i, row in enumerate(all_rows[1:], start=2):
        url = row[0].strip() if len(row) > 0 else ""
        status = row[1].strip() if len(row) > 1 else ""
        if url and status != "Listo" and ("youtube" in url or "youtu.be" in url):
            pending.append({"row": i, "url": url})
    if not pending:
        print("⚠️ No hay videos pendientes. Pega un link en la columna A de tu Sheet.")
    else:
        print(f"✅ [3/7] {len(pending)} video(s) pendiente(s)")
except Exception as e:
    print(f"❌ Error: {e}")
    print(f"   Asegurate de tener una Sheet llamada '{SHEET_NAME}'")
    pending = []

# ============================================================
# PASO 4: CARGAR WHISPER
# ============================================================
if pending:
    print("⏳ [4/7] Cargando Whisper (primera vez tarda ~1 min)...")
    import whisper
    whisper_model = whisper.load_model('base')
    print("✅ [4/7] Whisper listo")

# ============================================================
# FUNCIONES
# ============================================================
import edge_tts
from pydub import AudioSegment

async def tts_segment(text, path, voice, rate='+0%'):
    comm = edge_tts.Communicate(text, voice, rate=rate)
    await comm.save(path)

def get_duration(fp):
    r = subprocess.run(['ffprobe','-v','quiet','-print_format','json','-show_format',fp], capture_output=True, text=True)
    return float(json.loads(r.stdout)['format']['duration'])

def mix_audio_with_pydub(audio_files, video_duration_sec, output_wav):
    """Mezcla todos los audios en una sola pista usando pydub (sin limite de argumentos)."""
    # Crear pista de silencio de la duracion del video
    duration_ms = int(video_duration_sec * 1000)
    mixed = AudioSegment.silent(duration=duration_ms, frame_rate=44100)

    total = len(audio_files)
    for idx, af in enumerate(audio_files):
        try:
            seg_audio = AudioSegment.from_file(af['path'])
            position_ms = int(af['start'] * 1000)
            mixed = mixed.overlay(seg_audio, position=position_ms)
        except:
            pass
        if (idx + 1) % 500 == 0:
            print(f"   Mezclando audio: {idx+1}/{total}...")

    mixed.export(output_wav, format="wav")
    return output_wav

# ============================================================
# PROCESAR CADA VIDEO
# ============================================================
for item in pending:
    row_num = item['row']
    url = item['url']
    work = f'/content/work_{row_num}'
    os.makedirs(work, exist_ok=True)

    print(f"\n{'='*50}")
    print(f"🎬 Fila {row_num}: {url}")
    print(f"{'='*50}")
    sheet.update_cell(row_num, 2, 'Procesando...')

    try:
        # --- DESCARGAR ---
        print("⏳ [5/7] Descargando video...")
        vid_path = os.path.join(work, 'video.mp4')
        r = subprocess.run(['yt-dlp','-f','bestvideo[height<=720]+bestaudio/best[height<=720]',
                           '--merge-output-format','mp4','-o',vid_path,'--no-playlist',url],
                          capture_output=True, text=True)
        if r.returncode != 0:
            r = subprocess.run(['yt-dlp','-f','best[height<=720]','-o',vid_path,'--no-playlist',url],
                              capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError(f"yt-dlp fallo: {r.stderr[-200:]}")
        if not os.path.exists(vid_path):
            for f in os.listdir(work):
                if f.startswith('video'):
                    vid_path = os.path.join(work, f)
                    break
        print("✅ [5/7] Video descargado")

        # --- EXTRAER AUDIO ---
        wav_path = os.path.join(work, 'audio.wav')
        subprocess.run(['ffmpeg','-y','-i',vid_path,'-vn','-acodec','pcm_s16le',
                       '-ar','16000','-ac','1',wav_path], capture_output=True, check=True)

        # --- TRANSCRIBIR ---
        print("⏳ [6/7] Transcribiendo y generando voz...")
        result = whisper_model.transcribe(wav_path)
        segments = [{'id':i,'start':s['start'],'end':s['end'],'text':s['text'].strip()}
                    for i,s in enumerate(result['segments'])]
        lang = result.get('language','?')
        print(f"   {len(segments)} segmentos, idioma detectado: {lang}")

        # --- GENERAR TTS ---
        tts_dir = os.path.join(work, 'tts')
        os.makedirs(tts_dir, exist_ok=True)
        audio_files = []
        total_segs = len(segments)
        for seg in segments:
            txt = seg['text'].strip()
            if not txt:
                continue
            out = os.path.join(tts_dir, f"s{seg['id']:04d}.mp3")
            try:
                await tts_segment(txt, out, VOZ)
                audio_files.append({'path':out, 'start':seg['start']})
            except:
                pass
            if len(audio_files) % 500 == 0 and len(audio_files) > 0:
                print(f"   TTS: {len(audio_files)}/{total_segs} generados...")
        print(f"   {len(audio_files)} audios generados")
        print("✅ [6/7] Transcripcion y voz listas")

        # --- COMPONER VIDEO ---
        print("⏳ [7/7] Componiendo video final...")
        output_name = f"doblado_{int(time.time())}.mp4"
        output_path = os.path.join(work, output_name)

        if audio_files:
            vid_dur = get_duration(vid_path)
            # Usar pydub para mezclar (sin limite de argumentos)
            print(f"   Mezclando {len(audio_files)} audios con pydub...")
            mixed_audio = os.path.join(work, 'mixed_audio.wav')
            mix_audio_with_pydub(audio_files, vid_dur, mixed_audio)
            print("   Audio mezclado, componiendo video...")

            # Combinar video + audio mezclado
            subprocess.run(['ffmpeg','-y','-i',vid_path,'-i',mixed_audio,
                           '-c:v','copy','-c:a','aac','-b:a','192k',
                           '-map','0:v','-map','1:a','-shortest',output_path],
                          capture_output=True, check=True)
        else:
            shutil.copy2(vid_path, output_path)

        # --- SUBIR A DRIVE ---
        print("☁️ Subiendo a Google Drive...")
        drive_path = os.path.join(DRIVE_OUTPUT, output_name)
        shutil.copy2(output_path, drive_path)

        sheet.update_cell(row_num, 2, 'Listo')
        sheet.update_cell(row_num, 3, f'VideosTraducidos/{output_name}')

        print(f"✅ [7/7] Video listo!")
        print(f"\n🎉 VIDEO GUARDADO EN: Google Drive > VideosTraducidos > {output_name}")

        shutil.rmtree(work, ignore_errors=True)

    except Exception as e:
        print(f"\n❌ Error: {e}")
        sheet.update_cell(row_num, 2, f'Error: {str(e)[:80]}')

print(f"\n{'='*50}")
print("🏁 Terminado. Revisa tu Google Drive > VideosTraducidos")
print(f"{'='*50}")